# MobileNetV2 на CIFAR-10

Репликация MobileNetV2 (Sandler et al., 2018, arXiv:1801.04381) с двумя целями:

1. **Проверить блок библиотеки.** `InvertedResidual` из `spartan_torch` собирается в сеть
   по архитектурной таблице статьи (Table 2). Веса копируются из
   `torchvision.models.mobilenet_v2`, форварды сравниваются поэлементно — с key-remap,
   т.к. torchvision хранит слои в `Sequential` (`conv.0.0`, `conv.1.1`, ...), а наши блоки —
   явные подмодули (`conv1/bn1/...`).
2. **Трейн на CIFAR-10.** Рецепт статьи §6.1 (RMSProp, lr=0.045, weight_decay=4e-4)
   на малых разрешениях нестабилен: сеть не выходит из random (val_acc ~10% при любом lr RMSProp,
   проверено на стенде). Заменено на SGD lr=0.1 + warmup + cosine — как в resnet18-эксперименте,
   первый же прогон уходит от random уже на 150 шагах.

Сам `MobileNetV2` — код эксперимента (AGENTS.md: реимплементация стандартных моделей
в библиотеку не входит), в библиотеку попадают только переиспользуемые блоки.
`DepthwiseSeparableConv` — соседний примитив (MobileNetV1/SSDLite-стиль), в эту сеть не входит,
проверяется тестами.

Запуск: `uv sync --extra dev --extra experiments`, затем `uv run jupyter lab` и открыть
этот ноутбук. Cwd ноутбука = его папка, поэтому данные и чекпойнты лягут рядом с ним.


In [ ]:
# === Setup: среда + зависимости (локально / devcontainer / Colab) ===
import sys

IN_COLAB = "google.colab" in sys.modules
MLFLOW_ENABLED = not IN_COLAB  # в Colab локального MLflow-сервера нет

if IN_COLAB:
    import subprocess
    from pathlib import Path
    PROJECT_ROOT = Path("/content/spartan-torch")
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/mievst/spartan-torch.git", str(PROJECT_ROOT)],
            check=True,
        )
    # репо-модули экспериментов (tinyllama/*.py, vit/vision_transformer.py) + исходники lib
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
    # Colab приходит со своим numpy/scipy; наш -e ресолв мог рассогласовать их.
    # Апгрейдим пару вместе, чтобы они совпали (иначе datasets -> scipy падает
    # на numpy._core._multiarray_umath._blas_supports_fpe).
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "--upgrade-strategy", "eager", "numpy", "scipy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e",
         f"{PROJECT_ROOT}[experiments,dev]"],
        check=True,
    )
else:
    PROJECT_ROOT = None

print(f"IN_COLAB={IN_COLAB} | MLFLOW_ENABLED={MLFLOW_ENABLED} | PROJECT_ROOT={PROJECT_ROOT} | python={sys.version.split()[0]}")


## 0. Конфигурация

Пути и гиперпараметры — в одной ячейке. `data/` и `checkpoints/` уже в `.gitignore`,
в git уходит только ноутбук.


In [1]:
from pathlib import Path

import torch

ROOT = (PROJECT_ROOT / "experiments/image_classification/mobilenetv2") if IN_COLAB else Path.cwd()
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
DATA_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cwd: {ROOT} | device: {DEVICE}")

# --- гиперпараметры (SGD+cosine, как resnet18; про RMSProp см. §4) ---
EPOCHS = 100
BATCH_SIZE = 128
LR = 0.1            # начальный lr (SGD)
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 5   # линейный разогрев до LR
MIN_LR = 1e-4       # нижняя граница cosine
DROPOUT = 0.2       # dropout в классификаторе
NUM_WORKERS = 0       # Windows-safe
SEED = 0
PROFILER = "simple"   # None | "simple" | "advanced" | "pytorch"

torch.manual_seed(SEED)

# --- MLflow ---
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_EXPERIMENT_NAME = "mobilenetv2-cifar10"

torch.set_float32_matmul_precision("medium")


cwd: a:\projects\spartan-torch\experiments\image_classification\mobilenetv2 | device: cuda


## 1. Сборка MobileNetV2

Сеть собирается из `spartan_torch.InvertedResidual` по архитектурной таблице статьи (Table 2).
Каждая строка таблицы `(t, c, n, s)` — последовательность из `n` блоков с выходными каналами `c`,
expansion `t` и stride `s` только у первого блока в последовательности (остальные stride=1).

```
stem conv 3x3/2 -> 17x InvertedResidual -> conv 1x1 (1280) -> avgpool -> dropout 0.2 -> fc
```

`width_mult` (0.35–1.4 в статье) масштабирует каналы с округлением до кратного 8 —
иначе сеть с некратной шириной ломается на остаточных связях.

**Адаптация под 32×32.** Каноническая сеть рассчитана на 224×224: пять stride-2 стадий
сводят фиче-мап к 7×7. На CIFAR-10 (32×32) это дало бы схлопывание до **1×1** — depthwise-свёртки
перестают видеть пространство, сеть не учится (проверено: val_acc ~16% за 37 эпох).
Поэтому для CIFAR убираем две из четырёх stride-2 стадий (`for_cifar=True`), финал — **4×4**
(как в ResNet-CIFAR). Параметров не меняется, но с torchvision весами такая сеть уже не сверяется —
верификация ниже работает с канонической конфигурацией.


In [2]:
from collections import OrderedDict

import torch.nn.functional as F
from torch import nn

from spartan_torch import InvertedResidual


def _make_divisible(v, divisor=8, min_value=None):
    """Округляет каналы вверх до кратного divisor (точная копия torchvision).

    При width_mult < 1 держит ширину выровненной, чтобы остаточные связи
    сходились по числу каналов. При width_mult == 1 ничего не меняет.
    """
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


# Архитектурная таблица MobileNetV2 (Table 2 статьи), без stem и головы.
# Формат строки: (t, c, n, s) = expansion, каналы, число повторов, stride первой пары.
BOTTLENECK_SPECS = [
    (1, 16, 1, 1),
    (6, 24, 2, 2),
    (6, 32, 3, 2),
    (6, 64, 4, 2),
    (6, 96, 3, 1),
    (6, 160, 3, 2),
    (6, 320, 1, 1),
]


class MobileNetV2(nn.Module):
    """MobileNetV2, структурно идентичный torchvision.models.mobilenet_v2.

    features: stem (conv 3x3 stride 2) + 17 InvertedResidual + финальный
    conv 1x1 -> 1280. classifier: Dropout + Linear. avgpool — функциональный,
    как в torchvision, поэтому в state_dict не попадает.

    for_cifar: для входов 32x32 каноническая сеть схлопывает фиче-мап до 1x1
    (пять stride-2 стадий рассчитаны на 224x224). При for_cifar=True две из
    четырёх stride-2 стадий таблицы переводятся в stride 1, финал — 4x4.
    Количество параметров при этом не меняется.
    """

    def __init__(self, num_classes=1000, width_mult=1.0, expand_ratio=6,
                 dropout=0.2, norm_layer=nn.BatchNorm2d, for_cifar=False):
        super().__init__()

        def _cn(c):
            return _make_divisible(int(c * width_mult), 8)

        last_channel = _make_divisible(int(1280 * width_mult), 8)

        features = OrderedDict()
        features["0"] = nn.Sequential(OrderedDict([
            ("conv", nn.Conv2d(3, _cn(32), 3, stride=2, padding=1, bias=False)),
            ("bn", norm_layer(_cn(32))),
            ("relu6", nn.ReLU6(inplace=True)),
        ]))

        in_c = _cn(32)
        idx = 1
        for si, (t, c, n, s) in enumerate(BOTTLENECK_SPECS):
            # 32x32 слишком мал для полного даунсэмплинга (выход был бы 1x1):
            # убираем stride-2 у стадий (6,32,3,2) и (6,160,3,2) -> финал 4x4.
            s = 1 if (for_cifar and si in (2, 5)) else s
            out_c = _cn(c)
            for i in range(n):
                stride = s if i == 0 else 1
                features[str(idx)] = InvertedResidual(
                    in_c, out_c, stride=stride, expansion=t, norm_layer=norm_layer,
                )
                in_c = out_c
                idx += 1

        features[str(idx)] = nn.Sequential(OrderedDict([
            ("conv", nn.Conv2d(in_c, last_channel, 1, bias=False)),
            ("bn", norm_layer(last_channel)),
            ("relu6", nn.ReLU6(inplace=True)),
        ]))

        self.features = nn.Sequential(features)
        self.classifier = nn.Sequential(OrderedDict([
            ("dropout", nn.Dropout(p=dropout)),
            ("fc", nn.Linear(last_channel, num_classes)),
        ]))

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# Страховка: CIFAR-вариант обязан давать фиче-мап 4x4 на входе 32x32,
# а не схлопываться до 1x1, как каноническая сеть под 224x224.
_ck = MobileNetV2(num_classes=10, for_cifar=True)
with torch.no_grad():
    _feat = _ck.features(torch.randn(1, 3, 32, 32))
assert _feat.shape[-2:] == (4, 4), f"for_cifar feature map {tuple(_feat.shape)}, expected 4x4"
del _ck


## 2. Верификация против torchvision

`MobileNetV2` повторяет структуру `torchvision.models.mobilenet_v2`, но слои внутри блока
называются иначе: torchvision кладёт их в `Sequential` (`features.<i>.conv.0.0` = expand conv,
`conv.1.1` = dw bn, `conv.2` = project conv, `conv.3` = project bn), а наш блок — явные подмодули
`conv1/bn1/conv2/bn2/conv3/bn3`. `remap_tv_state_dict` транслирует ключи, `state_dict` грузится
со `strict=True`. Если форварды совпадут поэлементно — блоки эквивалентны.

Верифицируем **каноническую** конфигурацию (`for_cifar=False`) — только она совпадает
с torchvision. CIFAR-вариант (см. ячейку сборки) отклоняется от неё намеренно, он проверяется
`assert`-ом на 4×4 в той же ячейке.


In [3]:
import torchvision

vision = torchvision.models.mobilenet_v2(weights=None)
mine = MobileNetV2(num_classes=1000)


def remap_tv_state_dict(tv_sd, specs):
    """Транслирует state_dict torchvision в имена нашей сборки.

    torchvision: features.<i>.conv.<L>.<M>.<rest>, где
      - для блоков с expansion > 1: L=0 expand (Conv2dNormActivation),
        L=1 depthwise, project — плоские conv (L=2) и bn (L=3);
      - для блока с expansion == 1 expand-слоя нет: L=0 depthwise,
        project — conv (L=1) и bn (L=2);
      - M=0 свёртка, M=1 нормировка внутри Conv2dNormActivation.
    Наши блоки: conv1/bn1 (expand), conv2/bn2 (dw), conv3/bn3 (project).
    """
    block_exp = {}
    idx = 1
    for t, c, n, s in specs:
        for _ in range(n):
            block_exp[idx] = t
            idx += 1

    mapping = {}
    for key, value in tv_sd.items():
        parts = key.split(".")
        if parts[0] == "classifier":
            # classifier.1.* -> classifier.fc.* (classifier.0 — Dropout, без весов)
            if parts[1] == "1":
                mapping[f"classifier.fc.{'.'.join(parts[2:])}"] = value
            continue
        if parts[0] != "features":
            continue

        i = int(parts[1])
        if i not in block_exp:
            # stem (0) и финальный conv 1x1: Conv2dNormActivation (0=conv, 1=bn)
            layer = "conv" if parts[2] == "0" else "bn"
            mapping[f"features.{i}.{layer}.{'.'.join(parts[3:])}"] = value
            continue

        if len(parts) == 5:
            # плоский project: conv.<L>.<attr>
            L, attr = parts[3], parts[4]
            if (block_exp[i] > 1 and L == "2") or (block_exp[i] == 1 and L == "1"):
                mapping[f"features.{i}.conv3.{attr}"] = value
            else:
                mapping[f"features.{i}.bn3.{attr}"] = value
            continue

        # Conv2dNormActivation: conv.<L>.<M>.<rest> (M=0 conv, M=1 bn)
        L, M = parts[3], parts[4]
        rest = ".".join(parts[5:])
        if block_exp[i] > 1:
            conv_name, bn_name = ("conv1", "conv2")[int(L)], ("bn1", "bn2")[int(L)]
        else:
            conv_name, bn_name = "conv2", "bn2"
        name = conv_name if M == "0" else bn_name
        mapping[f"features.{i}.{name}.{rest}"] = value
    return mapping


state = remap_tv_state_dict(vision.state_dict(), BOTTLENECK_SPECS)
missing, unexpected = mine.load_state_dict(state, strict=True)
assert not missing and not unexpected
print(f"state_dict keys matched: {len(vision.state_dict())}")

vision.eval()
mine.eval()
x = torch.randn(4, 3, 32, 32)
with torch.no_grad():
    y_vision = vision(x)
    y_mine = mine(x)

max_diff = (y_mine - y_vision).abs().max().item()
assert torch.allclose(y_mine, y_vision, atol=1e-6), "FORWARD MISMATCH"
print(f"forward identical | max abs diff = {max_diff:.2e}")

n_vision = sum(p.numel() for p in vision.parameters())
n_mine = sum(p.numel() for p in mine.parameters())
assert n_vision == n_mine
print(f"params | torchvision: {n_vision:,} | ours: {n_mine:,}")


state_dict keys matched: 314
forward identical | max abs diff = 0.00e+00
params | torchvision: 3,504,872 | ours: 3,504,872


## 3. Данные: CIFAR-10

`CIFAR10DataModule` качает данные в `data/` папки эксперимента — полная изоляция между
экспериментами.


In [4]:
import lightning as L
import torchvision.transforms as T
from pathlib import Path

from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10


class CIFAR10DataModule(L.LightningDataModule):
    MEAN = (0.4914, 0.4822, 0.4465)
    STD = (0.2470, 0.2435, 0.2616)

    def __init__(self, data_dir, batch_size=128, num_workers=0):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.batch_size = batch_size
        self.num_workers = num_workers

    def prepare_data(self):
        CIFAR10(self.data_dir, train=True, download=True)
        CIFAR10(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        train_tf = T.Compose([
            T.RandomCrop(32, padding=4),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(self.MEAN, self.STD),
        ])
        val_tf = T.Compose([
            T.ToTensor(),
            T.Normalize(self.MEAN, self.STD),
        ])
        if stage in (None, "fit"):
            self.train_ds = CIFAR10(self.data_dir, train=True, transform=train_tf)
            self.val_ds = CIFAR10(self.data_dir, train=False, transform=val_tf)

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, batch_size=self.batch_size, shuffle=True,
            num_workers=self.num_workers, persistent_workers=self.num_workers > 0,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, persistent_workers=self.num_workers > 0,
        )


W0805 16:30:32.594000 7800 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


## 4. Обёртка Lightning

`configure_optimizers`: SGD + warmup + cosine (как resnet18-эксперимент). Сначала 5 эпох
линейного разогрева до `LR`, потом cosine-затухание до `MIN_LR` на остаток эпох.

Почему не RMSProp из статьи: на CIFAR-10 (32×32, без ImageNet-масштаба) он нестабилен —
lr 0.045 взрывает loss на первых шагах, при lr 0.045/0.01/0.003 val_acc остаётся на random
(~10%) за 150 шагов, тогда как SGD lr=0.1 за те же 150 шагов даёт 28%.


In [5]:
import torch.nn.functional as F
from torchmetrics import Accuracy

from spartan_torch import WarmupScheduler


class MobileNetV2Lit(L.LightningModule):
    def __init__(self, num_classes=10, lr=0.1, momentum=0.9, weight_decay=5e-4,
                 warmup_epochs=5, epochs=100, min_lr=1e-4, dropout=0.2, for_cifar=True):
        super().__init__()
        self.save_hyperparameters()
        self.model = MobileNetV2(num_classes=num_classes, dropout=dropout, for_cifar=for_cifar)
        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.train_acc(logits, y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_acc", self.train_acc, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.val_acc(logits, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.parameters(), lr=self.hparams.lr, momentum=self.hparams.momentum,
            weight_decay=self.hparams.weight_decay,
        )
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.hparams.epochs - self.hparams.warmup_epochs,
            eta_min=self.hparams.min_lr,
        )
        scheduler = WarmupScheduler(
            optimizer, warmup=self.hparams.warmup_epochs,
            scheduler=cosine, min_lr=self.hparams.min_lr,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "epoch", "frequency": 1},
        }


## 5. Коллбэки: графики, чекпойнты, MLflow

MLflow-логгер подключается только если сервер на `MLFLOW_TRACKING_URI` жив.


In [6]:
from lightning.pytorch.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename="mobilenetv2-{epoch:02d}-{val_acc:.3f}",
    monitor="val_acc",
    mode="max",
    save_top_k=1,
    save_last=True,
)

In [7]:
import socket
from urllib.parse import urlparse

from lightning.pytorch.loggers import MLFlowLogger


def _mlflow_reachable(uri: str, timeout: float = 2.0) -> bool:
    if not MLFLOW_ENABLED:
        return False
    parsed = urlparse(uri)
    try:
        with socket.create_connection((parsed.hostname, parsed.port), timeout=timeout):
            return True
    except OSError:
        return False


def make_logger():
    if not _mlflow_reachable(MLFLOW_TRACKING_URI):
        print(f"WARNING: MLflow недоступен ({MLFLOW_TRACKING_URI}) — работаем без логгера")
        return None
    return MLFlowLogger(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=MLFLOW_EXPERIMENT_NAME,
        save_dir=str(ROOT / "mlruns"),
    )


## 6. Тренировка

Рецепт: SGD lr=0.1, warmup 5 эпох, cosine до 1e-4, weight_decay 5e-4 (как resnet18).
100 эпох по умолчанию; для точности — увеличь `EPOCHS` в конфиге.


In [8]:
from lightning.pytorch import Trainer
from lightning.pytorch.profilers import AdvancedProfiler, PyTorchProfiler, SimpleProfiler


def make_profiler(name):
    if name in (None, "none"):
        return None
    if name == "simple":
        return SimpleProfiler(dirpath=str(CKPT_DIR), filename="profile")
    if name == "advanced":
        return AdvancedProfiler(dirpath=str(CKPT_DIR), filename="profile")
    if name == "pytorch":
        return PyTorchProfiler(dirpath=str(CKPT_DIR), filename="profile")
    raise ValueError(f"unknown profiler: {name}")

dm = CIFAR10DataModule(DATA_DIR, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
dm.prepare_data()  # качает CIFAR-10, если ещё нет

lit = MobileNetV2Lit(
    num_classes=10, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS, epochs=EPOCHS, min_lr=MIN_LR, dropout=DROPOUT,
)

logger = make_logger()

trainer = Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices="auto",
    callbacks=[checkpoint_cb],
    logger=logger,
    benchmark=True,
    precision="16-mixed",
    profiler=make_profiler(PROFILER),
    log_every_n_steps=20,
)

trainer.fit(lit, datamodule=dm)


a:\projects\spartan-torch\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
a:\projects\spartan-torch\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
LOCAL_RANK: 0 - CUDA_V

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ MobileNetV2        │  2.2 M │ train │     0 │
│ 1 │ train_acc │ MulticlassAccuracy │      0 │ train │     0 │
│ 2 │ val_acc   │ MulticlassAccuracy │      0 │ train │     0 │
└───┴───────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 2.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.2 M                                                                                                
Total estimated model params size (MB): 8.947                                                                      
Modules in train mode: 168                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

a:\projects\spartan-torch\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

a:\projects\spartan-torch\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

a:\projects\spartan-torch\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=100` reached.


🏃 View run sedate-yak-82 at: http://localhost:5000/#/experiments/3/runs/2b42817d986e48278522647c59b18c89
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [9]:
print(f"best ckpt: {checkpoint_cb.best_model_path}")
if logger is not None:
    print(f"MLflow run: {MLFLOW_TRACKING_URI}/#/experiments/{logger.experiment_id}/runs/{logger.run_id}")


best ckpt: A:\projects\spartan-torch\experiments\image_classification\mobilenetv2\checkpoints\mobilenetv2-epoch=97-val_acc=0.893.ckpt
MLflow run: http://localhost:5000/#/experiments/3/runs/2b42817d986e48278522647c59b18c89


## Итог

- `InvertedResidual` эквивалентен блоку torchvision — проверено через `state_dict` (strict)
  и поэлементное сравнение форвардов (max abs diff 0.0 при width_mult=1.0).
- `MobileNetV2` из блоков `spartan_torch` повторяет параметры torchvision
  (3 504 872 при 1000 классов).
- Для CIFAR-10 две stride-2 стадии убраны (`for_cifar=True`) — фиче-мап 4×4 вместо 1×1,
  иначе сеть не учится. Параметры те же, архитектура отклоняется от torchvision намеренно.
- Трейн идёт на SGD lr=0.1 + warmup + cosine (как resnet18): RMSProp из статьи §6.1
  на CIFAR-10 нестабилен и не выходит из random (val_acc ~10% при lr 0.045/0.01/0.003).
